# 2B - Create Streaming Tables with SQL using Auto Loader

In this demonstration we will create a streaming table to incrementally ingest files from a volume using Auto Loader with SQL.

When you create a streaming table using the `CREATE OR REFRESH STREAMING TABLE` statement, the initial data refresh and population begin immediately. These operations do not consume DBSQL warehouse compute. Instead, streaming table rely on serverless DLT for both creation and refresh. A dedicated serverless DLT pipeline is automatically created and managed by the system for each streaming table.

## Learning Objectives

By the end of this lesson, you should be able to:

- Create streaming tables in Databricks SQL for incremental data ingestion.
- Refresh streaming tables using the `REFRESH` statement.

## RECOMMENDATION

The `CREATE STREAMING TABLE` SQL command is the recommended alternative to the legacy `COPY INTO` SQL command for incremental ingestion from cloud object storage. Databricks recommends using streaming tables to ingest data using Databricks SQL.

A streaming table is a table registered to Unity Catalog with extra support for streaming or incremental data processing. A DLT pipeline is automatically created for each streaming table. You can use streaming tables for incremental data loading from Kafka and cloud object storage.

In [0]:
%sql
USE CATALOG azuredatabricks;
USE SCHEMA default;

# C. Create Streaming Tables for Incremental Processing

1. Complete the following to explore the volume `/Volumes/dbacademy/your-lab-user-schema/csv_files_autoloader_source` and confirm it contains a single CSV file.

   a. Select the **Catalog** icon on the left.  
   b. Expand the **dbacademy** catalog.  
   c. Expand your **labuser** schema.  
   d. Expand **Volumes**.  
   e. Expand the **csv_files_autoloader_source** volume.  
   f. Confirm it contains a single CSV file named `000.csv`.

2. Run the query below to view the data in the CSV file(s) in your cloud storage location. Notice that it was returned in tabular format and contains **3,149 rows**.

In [0]:
%sql
USE CATALOG azuredatabricks;
USE SCHEMA default;

## Creating Streaming Tables for Incremental Processing

In [0]:
%sql
SELECT *
FROM read_files('/Volumes/azuredatabricks/datasets/csv_files_autoloader_source',
format => 'csv',
sep => '|',
header => 'true'
);

## Create a STREAMING TABLE using Databricks SQL

3. Your goal is to create an incremental pipeline that only ingests new files (instead of using traditional batch ingestion). You can achieve this by using streaming tables in Databricks SQL (Auto Loader).

- The SQL code below creates a streaming table that will be scheduled to incrementally ingest only new data every week.
- A pipeline is automatically created for each streaming table. You can use streaming tables for incremental data loading from Kafka and cloud object storage.

**NOTE:** Incremental batch ingestion automatically detects new records in the data source and ignores records that have already been ingested. This reduces the amount of data processed, making ingestion jobs faster and more efficient in their use of compute resources.

**REQUIRED:** Please insert the path of your `csv_files_autoloader_source` volume in the `read_files` function. This process will take about a minute to run and set up the incremental ingestion pipeline.

In [0]:
%sql
CREATE OR REFRESH STREAMING TABLE sql_csv_autoloader
SCHEDULE EVERY 1 WEEK
AS
SELECT *
FROM STREAM read_files('/Volumes/azuredatabricks/datasets/csv_files_autoloader_source/',
format => 'csv',
sep => '|',
header => 'true'
);

In [0]:
%sql
SELECT *
FROM sql_csv_autoloader

In [0]:
%sql
DESCRIBE EXTENDED sql_csv_autoloader;

6. Describe the STREAMING TABLE and view the results. Notice the following:

- Under **Detailed Table Information**, notice the following rows:
  - **View Text**: The query that created the table.
  - **Type**: Specifies that it is a STREAMING TABLE.
  - **Provider**: Indicates that it is a Delta table.

- Under **Refresh Information**, you can see specific refresh details. Example shown below:

**Refresh Information**

7. The `DESCRIBE HISTORY` statement displays a detailed list of all changes, versions, and metadata associated with a Delta streaming table, including information on updates, deletions, and schema changes.

Run the cell below and view the results. Notice the following:

- In the **operation** column, you can see that a streaming table performs three operations: `CREATE TABLE`, `DLT SETUP`, and `STREAMING UPDATE`.

- Scroll to the right and find the **operationMetrics** column. In row 1 (Version 2 of the table), the value shows that the **numOutputRows** is 3149, indicating that 3149 rows were added to the `sql_csv_autoloader` table.

In [0]:
%sql
DESCRIBE HISTORY sql_csv_autoloader;

In [0]:
%sql
REFRESH STREAMING TABLE sql_csv_autoloader;

In [0]:
%sql
SELECT *
FROM sql_csv_autoloader

In [0]:
%sql
DESCRIBE HISTORY sql_csv_autoloader;

In [0]:
%sql
DROP TABLE IF EXISTS sql_csv_autoloader;

## Python Example

In [0]:
spark.sql("""
          LIST '/Volumes/azuredatabricks/datasets/csv_files_autoloader_source'""").display()

In [0]:
# Create a volume to store the Auto Loader checkpoint files
spark.sql(f'CREATE VOLUME IF NOT EXISTS azuredatabricks.default.auto_loader_files')

# Set checkpoint location to the volume from above
checkpoint_file_location = '/Volumes/azuredatabricks/default/auto_loader_files'

# Incrementally (or stream) data using Auto Loader
(
    spark
        .readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .option("sep", "|")
        .option("inferSchema", "true")
        .option("cloudFiles.schemaLocation", f"{checkpoint_file_location}")
        .load("/Volumes/azuredatabricks/datasets/csv_files_autoloader_source")
        .writeStream
        .option("checkpointLocation", f"{checkpoint_file_location}")
        .trigger(once=True)
        .toTable("azuredatabricks.default.python_csv_autoloader")
)

In [0]:
spark.sql("SELECT * FROM python_csv_autoloader").display()

In [0]:
from pyspark.sql.functions import *

In [0]:
files = dbutils.fs.ls("/Volumes/databricks_simulated_e_commerce_clickstream_data/v01/raw/sales-csv/")


for file in files[:2]:
    dbutils.fs.cp(file.path,
                  "/Volumes/azuredatabricks/datasets/csv_files_autoloader_source/" + file.name)

In [0]:
spark.sql("""
          LIST '/Volumes/azuredatabricks/datasets/csv_files_autoloader_source'""").display()

In [0]:
%sql
DESCRIBE HISTORY python_csv_autoloader;


In [0]:
%sql
DROP TABLE IF EXISTS sql_csv_autoloader;